In [1]:
import pandas as pd
import glob
import sys
import json


In [2]:
DATASET_GROUPS = {
    "centrifugal_pump": {
        "description": "Centrifugal pump device metadata and operating data.",
        "datasets": {
            "device_metadata": "generated_data/tabular/centrifugal_pump/centrifugal_pump_device_metadata_v1.parquet",
            "operating_data": "generated_data/tabular/centrifugal_pump/centrifugal_pump_operating_data_v1.parquet",
        },
    },
    "fraud_netherlands": {
        "description": "Imbalanced fraud detection data with customer profiles.",
        "datasets": {
            "customer_profiles": "generated_data/tabular/imbalanced_fraud/fraud_customer_profiles.parquet",
            "fraud_detection": "generated_data/tabular/imbalanced_fraud/imbalanced_fraud_detection.parquet",
        },
    },
    "sales_us_2019": {
        "description": "US retail sales data for 2019.",
        "datasets": {
            "sales_data": "generated_data/tabular/sales_data_2019/sales_data_2019.parquet",
        },
    },
    "wind_farm_no_repair": {
        "description": "Synthetic wind turbine stream and fault data. No repair events.",
        "datasets": {
            "stream": "generated_data/time_series/wind_data/wind_turbine_stream_data_without_repair.parquet",
            "fault_events": "generated_data/time_series/wind_data/wind_turbine_fault_events_without_repair.parquet",
        },
    },
    "wind_farm_with_repair": {
        "description": "Synthetic wind turbine stream and fault data. Includes repair events.",
        "datasets": {
            "stream": "generated_data/time_series/wind_data/wind_turbine_stream_data_with_repair.parquet",
            "fault_events": "generated_data/time_series/wind_data/wind_turbine_fault_events_with_repair.parquet",
        },
    },
    "solar_farm_no_repair": {
        "description": "Synthetic solar farm stream and fault data. No repair events.",
        "datasets": {
            "stream": "generated_data/time_series/solar_data/solar_farm_stream_data_without_repair.parquet",
            "fault_events": "generated_data/time_series/solar_data/solar_farm_fault_events_without_repair.parquet",
        },
    },
    "solar_farm_with_repair": {
        "description": "Synthetic solar farm stream and fault data. Includes repair events.",
        "datasets": {
            "stream": "generated_data/time_series/solar_data/solar_farm_stream_data_with_repair.parquet",
            "fault_events": "generated_data/time_series/solar_data/solar_farm_fault_events_with_repair.parquet",
        },
    },
}

BASE_PATH = "/Users/aniket/github/synthetic_data_generator/"

In [3]:
def load_group(group: dict, base_path: str) -> dict[str, pd.DataFrame]:
    return {
        name: pd.read_parquet(base_path + path)
        for name, path in group["datasets"].items()
    }


def load_all(groups: dict, base_path: str) -> dict[str, dict[str, pd.DataFrame]]:
    return {
        group_name: load_group(group, base_path)
        for group_name, group in groups.items()
    }

In [4]:
loaded = load_all(DATASET_GROUPS, BASE_PATH)

In [5]:
sys.getsizeof(loaded)

272

In [6]:
total_bytes = sum(
    df.memory_usage(deep=True).sum() 
    for sub_dict in loaded.values() 
    for df in sub_dict.values()
)

def format_size(bytes_val):
    for unit in ['B', 'KB', 'MB', 'GB']:
        if bytes_val < 1024:
            return f"{bytes_val:.2f} {unit}"
        bytes_val /= 1024
    return f"{bytes_val:.2f} TB"

print(f"Total memory: {format_size(total_bytes)}")

Total memory: 919.03 MB


In [7]:
def dtype_to_kind(dtype) -> str:
    name = str(dtype)
    if "int" in name: return "int"
    if "float" in name: return "float"
    if "bool" in name: return "bool"
    if "datetime" in name or "timestamp" in name: return "dt"
    if "object" in name or "string" in name: return "obj"
    return "other"


def build_ds_entry(df: pd.DataFrame) -> dict:
    null_pct = round(df.isnull().mean().mean() * 100, 1)
    schema = [{"name": col, "type": dtype_to_kind(df[col].dtype)} for col in df.columns]
    sample = df.head(3).astype(str).replace("NaT", "NaT").to_dict(orient="records")
    return {"rows": len(df), "cols": len(df.columns), "nullPct": null_pct, "schema": schema, "sample": sample}


def build_explorer_data(groups: dict, loaded: dict) -> dict:
    return {
        group_name: {
            "desc": groups[group_name]["description"],
            "datasets": {
                ds_name: build_ds_entry(df)
                for ds_name, df in ds_dict.items()
            }
        }
        for group_name, ds_dict in loaded.items()
    }

In [15]:
explorer_data = build_explorer_data(DATASET_GROUPS, loaded)
# print(json.dumps(explorer_data, default=str, indent=2))

In [16]:
import openml
import pandas as pd

openml.config.apikey = "YOUR_API_KEY"  # from openml.org/settings

def upload_to_openml(df: pd.DataFrame, name: str, description: str) -> int:
    dataset = openml.datasets.OpenMLDataset(
        name=name,
        description=description,
        data=df,
        format="ARFF",          # openml converts from pandas internally
        licence="CC BY 4.0",
        default_target_attribute=None,  # set this if there is a label column
    )
    dataset.publish()
    return dataset.dataset_id

ModuleNotFoundError: No module named 'openml'

In [9]:
# # centrifugal pump
# pump_dev_metadata = pd.read_parquet("/Users/aniket/github/synthetic_data_generator/generated_data/tabular/centrifugal_pump/centrifugal_pump_device_metadata_v1.parquet")
# pump_data = pd.read_parquet("/Users/aniket/github/synthetic_data_generator/generated_data/tabular/centrifugal_pump/centrifugal_pump_operating_data_v1.parquet")

# # fraud data, netherlands
# fraud_customers = pd.read_parquet("/Users/aniket/github/synthetic_data_generator/generated_data/tabular/imbalanced_fraud/fraud_customer_profiles.parquet")
# fraid_data = pd.read_parquet("/Users/aniket/github/synthetic_data_generator/generated_data/tabular/imbalanced_fraud/imbalanced_fraud_detection.parquet")

# # sales data US
# sales_data = pd.read_parquet("/Users/aniket/github/synthetic_data_generator/generated_data/tabular/sales_data_2019/sales_data_2019.parquet")

# # synthetic wind farm data without repair
# wind_no_repair_fault = pd.read_parquet("/Users/aniket/github/synthetic_data_generator/generated_data/time_series/wind_data/wind_turbine_fault_events_without_repair.parquet")
# wind_no_repair_stream = pd.read_parquet("/Users/aniket/github/synthetic_data_generator/generated_data/time_series/wind_data/wind_turbine_stream_data_without_repair.parquet")

# # synthetic wind farm data with repair
# wind_with_repair_fault = pd.read_parquet("/Users/aniket/github/synthetic_data_generator/generated_data/time_series/wind_data/wind_turbine_fault_events_with_repair.parquet")
# wind_with_repair_stream = pd.read_parquet("/Users/aniket/github/synthetic_data_generator/generated_data/time_series/wind_data/wind_turbine_stream_data_with_repair.parquet")

# # synthetic solar farm data without repair
# solar_no_repair_fault = pd.read_parquet("/Users/aniket/github/synthetic_data_generator/generated_data/time_series/solar_data/solar_farm_fault_events_without_repair.parquet")
# solar_no_repair_stream = pd.read_parquet("/Users/aniket/github/synthetic_data_generator/generated_data/time_series/solar_data/solar_farm_stream_data_without_repair.parquet")

# # synthetic solar farm data with repair
# solar_with_repair_fault = pd.read_parquet("/Users/aniket/github/synthetic_data_generator/generated_data/time_series/solar_data/solar_farm_fault_events_with_repair.parquet")
# solar_with_repair_stream = pd.read_parquet("/Users/aniket/github/synthetic_data_generator/generated_data/time_series/solar_data/solar_farm_stream_data_with_repair.parquet")

In [10]:
# wind_with_repair_fault

In [11]:
# wind_with_repair_stream

In [12]:
# datasets = {
#     "pump_device_metadata": "/Users/aniket/github/synthetic_data_generator/generated_data/tabular/centrifugal_pump/centrifugal_pump_device_metadata_v1.parquet",
#     "pump_operating_data": "/Users/aniket/github/synthetic_data_generator/generated_data/tabular/centrifugal_pump/centrifugal_pump_operating_data_v1.parquet",
#     "fraud_customer_profiles": "/Users/aniket/github/synthetic_data_generator/generated_data/tabular/imbalanced_fraud/fraud_customer_profiles.parquet",
#     "fraud_detection": "/Users/aniket/github/synthetic_data_generator/generated_data/tabular/imbalanced_fraud/imbalanced_fraud_detection.parquet",
#     "sales_data_2019": "/Users/aniket/github/synthetic_data_generator/generated_data/tabular/sales_data_2019/sales_data_2019.parquet",
#     "wind_no_repair_faults": "/Users/aniket/github/synthetic_data_generator/generated_data/time_series/wind_data/wind_turbine_fault_events_without_repair.parquet",
#     "wind_no_repair_stream": "/Users/aniket/github/synthetic_data_generator/generated_data/time_series/wind_data/wind_turbine_stream_data_without_repair.parquet",
#     "wind_with_repair_faults": "/Users/aniket/github/synthetic_data_generator/generated_data/time_series/wind_data/wind_turbine_fault_events_with_repair.parquet",
#     "wind_with_repair_stream": "/Users/aniket/github/synthetic_data_generator/generated_data/time_series/wind_data/wind_turbine_stream_data_with_repair.parquet",
#     "solar_no_repair_faults": "/Users/aniket/github/synthetic_data_generator/generated_data/time_series/solar_data/solar_farm_fault_events_without_repair.parquet",
#     "solar_no_repair_stream": "/Users/aniket/github/synthetic_data_generator/generated_data/time_series/solar_data/solar_farm_stream_data_without_repair.parquet",
#     "solar_with_repair_faults": "/Users/aniket/github/synthetic_data_generator/generated_data/time_series/solar_data/solar_farm_fault_events_with_repair.parquet",
#     "solar_with_repair_stream": "/Users/aniket/github/synthetic_data_generator/generated_data/time_series/solar_data/solar_farm_stream_data_with_repair.parquet",
# }

In [13]:
# def load_datasets(dataset_paths: dict[str, str]) -> dict[str, pd.DataFrame]:
#     return {name: pd.read_parquet(path) for name, path in dataset_paths.items()}


# def summarise_dataset(name: str, df: pd.DataFrame) -> None:
#     print(f"\n{'='*60}")
#     print(f"Dataset: {name}")
#     print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
#     print(f"\nColumn types:\n{df.dtypes}")
#     print(f"\nSample (first 3 rows):\n{df.head(3)}")
#     print(f"\nNull counts:\n{df.isnull().sum()}")
#     print(f"\nBasic stats:\n{df.describe(include='all')}")


# def explore_all(loaded: dict[str, pd.DataFrame]) -> None:
#     for name, df in loaded.items():
#         summarise_dataset(name, df)

In [14]:
# loaded_datasets = load_datasets(datasets)
# explore_all(loaded_datasets)